# model training

In [2]:
import pandas as pd
data = pd.read_csv('../data.csv')
print(data.to_string)

<bound method DataFrame.to_string of       age  gender    status  pressurehight  pressurelow  glucose    kcm  \
0      64       1  negative            160           83    160.0   1.80   
1      21       1  positive             98           46    296.0   6.75   
2      55       1  negative            160           77    270.0   1.99   
3      64       1  positive            120           55    270.0  13.87   
4      55       1  negative            112           65    300.0   1.08   
...   ...     ...       ...            ...          ...      ...    ...   
1314   44       1  negative            122           67    204.0   1.63   
1315   66       1  positive            125           55    149.0   1.33   
1316   45       1  positive            168          104     96.0   1.24   
1317   54       1  positive            117           68    443.0   5.80   
1318   51       1  positive            157           79    134.0  50.89   

      troponin  impluse  
0        0.012       66  
1        1

### Data processing

In [3]:
x=data.drop('status',axis='columns')
y=data['status']

In [4]:
x.columns

Index(['age', 'gender', 'pressurehight', 'pressurelow', 'glucose', 'kcm',
       'troponin', 'impluse'],
      dtype='object')

### split data 

In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)  # "negative" -> 0, "positive" -> 1
y_test_encoded = le.transform(y_test)  # "negative" -> 0, "positive" -> 1

# Pipeline 

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
num_cols=['age', 'gender', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin', 'impluse']
preprocessor=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),num_cols)
    ]
)
pipeline=Pipeline(steps=[
('preprocessor',preprocessor),
('model',RandomForestClassifier(n_estimators=100, random_state=42))
])
param_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [None, 10, 20]
}
gs = GridSearchCV(pipeline,
                  param_grid,
                  cv=5, 
                  scoring="accuracy",
                  n_jobs=-1)
gs.fit(X_train, y_train_encoded)

print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)
y_pred = gs.best_estimator_.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test_encoded, y_pred))
print("Test Precision:", precision_score(y_test_encoded, y_pred))
print("Test Recall:", recall_score(y_test_encoded, y_pred))
print("Test F1:", f1_score(y_test_encoded, y_pred))


Best params: {'model__max_depth': None, 'model__n_estimators': 50}
Best score: 0.9876777251184834
Test Accuracy: 0.9848484848484849
Test Precision: 0.9877300613496932
Test Recall: 0.9877300613496932
Test F1: 0.9877300613496932


# Saving model using Joblib

In [ ]:
import joblib

joblib.dump(gs.best_estimator_, "cardio_model.pkl")
print("✅ Pipeline saved successfully!")

✅ Pipeline saved successfully!
